In [1]:
#import necessary modules
import json
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

In [2]:
!curl http://localhost:9200/

{
  "name" : "LAPTOP-UDLON1P4",
  "cluster_name" : "elasticsearch",
  "cluster_uuid" : "6eRXGDVkRza-7IQXW6AsSg",
  "version" : {
    "number" : "9.2.3",
    "build_flavor" : "default",
    "build_type" : "zip",
    "build_hash" : "d8972a71dbbd64ff17f2f4dba9ca2c3fe09fb100",
    "build_date" : "2025-12-16T10:09:08.849001802Z",
    "build_snapshot" : false,
    "lucene_version" : "10.3.2",
    "minimum_wire_compatibility_version" : "8.19.0",
    "minimum_index_compatibility_version" : "8.0.0"
  },
  "tagline" : "You Know, for Search"
}


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100    539 100    539   0      0  42109      0                              0
100    539 100    539   0      0  41737      0                              0
100    539 100    539   0      0  41397      0                              0


In [3]:
# instantiate Elasticsearch client localhost
client = Elasticsearch("http://localhost:9200")

In [4]:
import pandas as pd


In [5]:
## reading the documents file to make a jsonl file
docs = pd.read_csv("IR2025/documents.csv")
print(docs.head())

## creating a jsonl file from documents.csv for elasticsearch
with open("documents.jsonl", "w", encoding = "utf-8") as file:
    for _, row in docs.iterrows():
        doc = {
            "doc_id" : row["ID"],
            "text" : row["Text"]
        }
        file.write(json.dumps(doc, ensure_ascii=False) + "\n")

       ID                                               Text
0  193157  Support towards the Europe PMC initiative-Cont...
1  193158  Support to the Vice-Presidents of the ERC Scie...
2  193159  Implementation of activities described in the ...
3  193160  Monitoring Atmospheric Composition and Climate...
4  193161  Pre-Operational Marine Service Continuity in T...


In [6]:
## to make sure the jsonl file is created correctly
with open("documents.jsonl", "r", encoding = "utf-8") as file:  
    for i in range(3):
        print(file.readline())

{"doc_id": 193157, "text": "Support towards the Europe PMC initiative-Contribution for 2014-2016: \"The proposed action will provide continued support to the European Research Council (ERC) in the implementation of its Open Access strategy for projects funded in the Life Sciences domain. It follows on from the project \"\"Support towards the Europe PMC initiative-Contribution for 2013\"\"(ERC-EuropePMC-SUP-2013) which has allowed the ERC to offer the benefits of Europe PMC to its funded researchers for the first time in 2013. The ERC Open Access strategy, and how the present project will assist the ERC in its implementation, is explained below\""}

{"doc_id": 193158, "text": "Support to the Vice-Presidents of the ERC Scientific Council 2014: The proposed Action will provide the necessary support to the Vice-Presidents of the European Research Council Scientific Council (ERC ScC) to achieve key milestones and deliverables of the ScC, which are required in the first year of implementatio

In [7]:
# read jsonl file line by line and put the records in a list to use later

all_json_data = [] # a list of json records
with open("documents.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        json_data = json.loads(line)  ## one line is one record/document
        all_json_data.append(json_data)

In [8]:
# take a peek at the data
print(f" Length of the list: {len(all_json_data)}")  ## 18316
#all_json_data[0]

for jd in all_json_data[:5]:
    print(f"DOC ID: {jd['doc_id']}, \nTEXT: {jd['text']} \n")

 Length of the list: 18316
DOC ID: 193157, 
TEXT: Support towards the Europe PMC initiative-Contribution for 2014-2016: "The proposed action will provide continued support to the European Research Council (ERC) in the implementation of its Open Access strategy for projects funded in the Life Sciences domain. It follows on from the project ""Support towards the Europe PMC initiative-Contribution for 2013""(ERC-EuropePMC-SUP-2013) which has allowed the ERC to offer the benefits of Europe PMC to its funded researchers for the first time in 2013. The ERC Open Access strategy, and how the present project will assist the ERC in its implementation, is explained below" 

DOC ID: 193158, 
TEXT: Support to the Vice-Presidents of the ERC Scientific Council 2014: The proposed Action will provide the necessary support to the Vice-Presidents of the European Research Council Scientific Council (ERC ScC) to achieve key milestones and deliverables of the ScC, which are required in the first year of imp

In [9]:
client.indices.delete(index="my_idx")   ## delete if previous index exists

ObjectApiResponse({'acknowledged': True})

In [10]:
# set similarity function to bm25                             
# set analyzer to english analyzer (stopword removal, stemming, puncuation removal, etc.) for indexing and querying

my_mapping = {
    "settings": {     
        "number_of_shards": 1,    ## εξαρταται απο μεγεθος ευρετηριου/nodes/ογκος κειμενων στα κλαστερς κλπ..
        "similarity": {           
            "my_bm25": {
                "type": "BM25",
                "k1": 1.2,
                "b": 0.75
            }
        },        
        "analysis": {     ## english analyzer
            "analyzer": {
                "default": {  ## για την ευρετηριαση 
                    "type": "english"
                },
                "default_search": {  ## για την αναζητηση
                    "type": "english"
                }
            }
        }
    },    
    "mappings": {
        "properties": {
            "doc_id": {"type": "integer"},
            "text": {"type": "text",  "similarity": "my_bm25"}
        }
    }
}

# create an index with the configuration above
client.indices.create(index='my_idx', body=my_mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_idx'})

In [11]:
exists = client.indices.exists(index="my_idx")
print("Index exists: ", exists)

mapp= client.indices.get_mapping(index="my_idx")
print(mapp["my_idx"]["mappings"])

Index exists:  True
{'properties': {'doc_id': {'type': 'integer'}, 'text': {'type': 'text', 'analyzer': 'default', 'similarity': 'my_bm25'}}}


In [12]:
# bulk insert documents in index using bulk helpers 
from elasticsearch.helpers import bulk

documents = []
for doc in all_json_data:
    documents.append({
        "_index": "my_idx",
        "_source": doc  # αν το doc είναι dict
    })

# Indexing με helpers.bulk
bulk(client, documents, refresh=True)

print("Done indexing documents into `my_idx` index!")

Done indexing documents into `my_idx` index!


In [13]:
def pretty_search_response(response): ## prints the search response
    if len(response["hits"]["hits"]) == 0:
        print("Your search returned no results.")
    else:
        for hit in response["hits"]["hits"]:
            id = hit["_id"]   ## ID που εφτιαξε η elasticsearch 
            score = hit["_score"]  
            doc_id = hit["_source"]["doc_id"]
            text = hit["_source"]["text"]
            
            pretty_output = f"\nID: {id}\nScore: {score}\nDoc ID: {doc_id}\nText: {text}\n"

            print(pretty_output)

In [14]:
search_results = client.search(index="my_movies", query={"match_all": {}})  

# Print search results
pretty_search_response(search_results)



ID: k2_YlJsBS7-TIK4up_cV
Score: 1.0
Doc ID: 204380
Text: A disruptive innovation for the minimisation of railway maintenance costs: Rail infrastructure managers in Europe invest yearly over €869 Million (from public funds) in repairing railway track damages caused by the poor conditions of train wheels. Nowadays, the most advanced systems to monitor the conditions of the train wheels (WID or Wheel Impact Detectors) to prevent/ minimise the damage produced to the railway track cannot anticipate accurately the occurrence of the failure or indicate to the train operating companies which maintenance operations must be performed and when it is the optimum moment. AMINSA have developed DTD System, an innovative system capable of detecting with high precision wheel defects at the onset of deterioration, and using this information to design a predictive maintenance plan. The added value of DTD SYSTEM compared to its competitors is based on: - Its superior accuracy thanks to the new filtering 

In [15]:
## testing a search
search_results = client.search(
    index="my_idx",
    size=5,                              
    query = {"match": {"text": "neurostimulation"}},   
)

# Print search results
pretty_search_response(search_results) 


ID: aT47j54BQuA2dDEUgOOu
Score: 12.3726845
Doc ID: 195939
Text: Understanding Emotional Actions: Emotions shape our personal lives and society at large. In Europe emotional disorders affect one in five people; it is the most chronic and second to most disabling health condition. The stakes to better understand our emotional behaviour are high; however there is still a lot to gain. Traditionally, emotions are thought to guide our behaviour either through fast instincts or deliberately selected actions. Current explanations of emotional health and disease frequently focus on one of these single components in isolation (or regard them as exclusive antagonists), but likely underestimate our emotional behaviour’s complexity and diversity. Here I will investigate a complementary account: I suggest that in humans optimal emotional behaviour arises not from these instinctual and deliberate action selection systems individually, but from their interaction, mediated in our brain by a likely uni

In [16]:
## dl (document length after tokenization) = 184 for document 195939 (1st place)

res = client.search(
    index="my_idx",
    query={
        "term": {
            "doc_id": 195939
        }
    }
)

print(res["hits"]["hits"][0]["_id"])

client.explain(
    index="my_idx",
    id=res["hits"]["hits"][0]["_id"],
    body={
        "query": {
            "match": {
                "text": "neurostimulation"
            }
        }
    }
)

aT47j54BQuA2dDEUgOOu


ObjectApiResponse({'_index': 'my_idx', '_id': 'aT47j54BQuA2dDEUgOOu', 'matched': True, 'explanation': {'value': 12.3726845, 'description': 'weight(text:neurostimul in 3267) [PerFieldSimilarity], result of:', 'details': [{'value': 12.3726845, 'description': 'score(freq=3.0), computed as boost * idf * tf from:', 'details': [{'value': 2.2, 'description': 'boost', 'details': []}, {'value': 7.800682, 'description': 'idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:', 'details': [{'value': 7, 'description': 'n, number of documents containing term', 'details': []}, {'value': 18316, 'description': 'N, total number of documents with field', 'details': []}]}, {'value': 0.72095585, 'description': 'tf, computed as freq / (freq + k1 * (1 - b + b * dl / avgdl)) from:', 'details': [{'value': 3.0, 'description': 'freq, occurrences of term within document', 'details': []}, {'value': 1.2, 'description': 'k1, term saturation parameter', 'details': []}, {'value': 0.75, 'description': 'b, length no

In [17]:
## dl (document length after tokenization) = 200 for document 204389 (2nd place)
res = client.search(
    index="my_idx",
    query={
        "term": {
            "doc_id": 204389
        }
    }
)

print(res["hits"]["hits"][0]["_id"])

client.explain(
    index="my_idx",
    id=res["hits"]["hits"][0]["_id"],
    body={
        "query": {
            "match": {
                "text": "neurostimulation"
            }
        }
    }
)

1z47j54BQuA2dDEUr_nT


ObjectApiResponse({'_index': 'my_idx', '_id': '1z47j54BQuA2dDEUr_nT', 'matched': True, 'explanation': {'value': 12.153968, 'description': 'weight(text:neurostimul in 9) [PerFieldSimilarity], result of:', 'details': [{'value': 12.153968, 'description': 'score(freq=3.0), computed as boost * idf * tf from:', 'details': [{'value': 2.2, 'description': 'boost', 'details': []}, {'value': 7.800682, 'description': 'idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:', 'details': [{'value': 7, 'description': 'n, number of documents containing term', 'details': []}, {'value': 18316, 'description': 'N, total number of documents with field', 'details': []}]}, {'value': 0.7082112, 'description': 'tf, computed as freq / (freq + k1 * (1 - b + b * dl / avgdl)) from:', 'details': [{'value': 3.0, 'description': 'freq, occurrences of term within document', 'details': []}, {'value': 1.2, 'description': 'k1, term saturation parameter', 'details': []}, {'value': 0.75, 'description': 'b, length normaliz

In [18]:
## reading the queries file
queries = pd.read_csv("IR2025/queries.csv")
print(queries.head())

    ID                                               Text
0  Q01  EUTRAVEL Optimodal European Travel Ecosystem E...
1  Q02  Track And Know Big Data for Mobility Tracking ...
2  Q03  SELIS, Towards a Shared European Logistics Int...
3  Q04  TYPHON Polyglot and Hybrid Persistence Archite...
4  Q05  CHARIOT Cognitive Heterogeneous Architecture f...


In [19]:
## method to run all the queries for k  retrieved documents

def queries_run(client, queries, k):
    results = []
    for _, row in queries.iterrows():
        Qid = row["ID"]      ## from queries file
        Qtext = row["Text"]

        search_results = client.search(    ## free text - match query text to document text
            index = "my_idx",
            size = k,
            query = {
                "match" : {
                    "text": Qtext
                }
            }
        )

        for rank, hit in enumerate(search_results["hits"]["hits"], start=1):
            
            doc_id = hit["_source"]["doc_id"]   ## the document id , not id from elasticsearch
            score = hit["_score"]    ## similarity score

            ## format results for trec_eval 
            results.append({    ## put every retrieved document in results list to later put in a file 
                "query_id" : Qid,
                "doc_id" : doc_id,
                "rank" : 0,
                "score" : score
                
            })

    return results    

In [20]:
results_20 = queries_run(client, queries, k=20)
print(results_20)
results_30 = queries_run(client, queries, k=30)
results_50 = queries_run(client, queries, k=50)

[{'query_id': 'Q01', 'doc_id': 193378, 'rank': 0, 'score': 764.02673}, {'query_id': 'Q01', 'doc_id': 193373, 'rank': 0, 'score': 273.72284}, {'query_id': 'Q01', 'doc_id': 205685, 'rank': 0, 'score': 244.1519}, {'query_id': 'Q01', 'doc_id': 193715, 'rank': 0, 'score': 242.06233}, {'query_id': 'Q01', 'doc_id': 206230, 'rank': 0, 'score': 222.9547}, {'query_id': 'Q01', 'doc_id': 193353, 'rank': 0, 'score': 222.53535}, {'query_id': 'Q01', 'doc_id': 193375, 'rank': 0, 'score': 220.6753}, {'query_id': 'Q01', 'doc_id': 193386, 'rank': 0, 'score': 216.5555}, {'query_id': 'Q01', 'doc_id': 211346, 'rank': 0, 'score': 215.89311}, {'query_id': 'Q01', 'doc_id': 202703, 'rank': 0, 'score': 213.50832}, {'query_id': 'Q01', 'doc_id': 206228, 'rank': 0, 'score': 212.71326}, {'query_id': 'Q01', 'doc_id': 213250, 'rank': 0, 'score': 212.42722}, {'query_id': 'Q01', 'doc_id': 194660, 'rank': 0, 'score': 210.28812}, {'query_id': 'Q01', 'doc_id': 210137, 'rank': 0, 'score': 207.94757}, {'query_id': 'Q01', 'do

In [21]:
## put results in a file for trec eval

def file_results(results, filename, run_name):
    with open(filename, "w") as file:
        for result in results:
            file.write(
                f"{result['query_id']} Q0 {result['doc_id']} "
                f"{result['rank']} {result['score']} {run_name} \n "
            )

In [22]:
file_results(results_20, "run_bm25_k20.txt", "bm25_k20")
file_results(results_30, "run_bm25_k30.txt", "bm25_k30")
file_results(results_50, "run_bm25_k50.txt", "bm25_k50")

In [23]:
with open("trec_eval/run_bm25_k20.txt") as file:
    print(file.read())



Q01 Q0 193378 0 764.02673 bm25_k20 
 Q01 Q0 193373 0 273.72284 bm25_k20 
 Q01 Q0 205685 0 244.1519 bm25_k20 
 Q01 Q0 193715 0 242.06233 bm25_k20 
 Q01 Q0 206230 0 222.9547 bm25_k20 
 Q01 Q0 193353 0 222.53535 bm25_k20 
 Q01 Q0 193375 0 220.6753 bm25_k20 
 Q01 Q0 193386 0 216.5555 bm25_k20 
 Q01 Q0 211346 0 215.89311 bm25_k20 
 Q01 Q0 202703 0 213.50832 bm25_k20 
 Q01 Q0 206228 0 212.71326 bm25_k20 
 Q01 Q0 213250 0 212.42722 bm25_k20 
 Q01 Q0 194660 0 210.28812 bm25_k20 
 Q01 Q0 210137 0 207.94757 bm25_k20 
 Q01 Q0 213278 0 206.8191 bm25_k20 
 Q01 Q0 205643 0 206.60056 bm25_k20 
 Q01 Q0 211970 0 204.75098 bm25_k20 
 Q01 Q0 211972 0 199.05475 bm25_k20 
 Q01 Q0 206824 0 197.84494 bm25_k20 
 Q01 Q0 193402 0 197.28416 bm25_k20 
 Q02 Q0 213164 0 520.1175 bm25_k20 
 Q02 Q0 210232 0 202.63286 bm25_k20 
 Q02 Q0 194301 0 155.4609 bm25_k20 
 Q02 Q0 194787 0 147.38472 bm25_k20 
 Q02 Q0 213660 0 142.36723 bm25_k20 
 Q02 Q0 210106 0 139.49901 bm25_k20 
 Q02 Q0 196653 0 133.73792 bm25_k20 
 Q02 Q0 2